# LLaVA-1.5-7b Demo

This notebook demonstrates how to use the LLaVA-1.5-7b model to answer questions about images sourced from the web.

In [ ]:
import torch
import requests
from PIL import Image
from io import BytesIO
from utils.model_utils import load_hf_model_and_processor_or_tokenizer

# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
model_name = "llava-hf/llava-1.5-7b-hf"
print(f"Loading model: {model_name}...")

# Use the utility function to load the model and processor
model, processor = load_hf_model_and_processor_or_tokenizer(
    model_name_or_path=model_name,
    device_map=device,
    dtype=torch.bfloat16,
    attn_implementation='sdpa'
)

print("Model loaded successfully.")

In [ ]:
import torch
import matplotlib.pyplot as plt
from typing import Unpack
from transformers.cache_utils import Cache, DynamicCache
from transformers.utils import TransformersKwargs
from transformers.modeling_outputs import BaseModelOutputWithPast
from transformers.models.llava.modeling_llava import LlavaModelOutputWithPast
from transformers.masking_utils import create_causal_mask
from functools import partialmethod
from transformers import LlamaModel, LlavaModel
torch.set_printoptions(threshold=10_000)


def llava_forward(
    self,
    input_ids: torch.LongTensor | None = None,
    pixel_values: torch.FloatTensor | None = None,
    attention_mask: torch.Tensor | None = None,
    position_ids: torch.LongTensor | None = None,
    past_key_values: Cache | None = None,
    inputs_embeds: torch.FloatTensor | None = None,
    vision_feature_layer: int | list[int] | None = None,
    vision_feature_select_strategy: str | None = None,
    cache_position: torch.LongTensor | None = None,
    image_sizes: torch.Tensor | None = None,
    **kwargs: Unpack[TransformersKwargs],
) -> tuple | LlavaModelOutputWithPast:
    vision_feature_layer = (
        vision_feature_layer
        if vision_feature_layer is not None
        else self.config.vision_feature_layer
    )
    vision_feature_select_strategy = (
        vision_feature_select_strategy
        if vision_feature_select_strategy is not None
        else self.config.vision_feature_select_strategy
    )

    if (input_ids is None) ^ (inputs_embeds is not None):
        raise ValueError("You must specify exactly one of input_ids or inputs_embeds")

    if inputs_embeds is None:
        inputs_embeds = self.get_input_embeddings()(input_ids)

    if pixel_values is not None:
        image_features = self.get_image_features(
            pixel_values=pixel_values,
            vision_feature_layer=vision_feature_layer,
            vision_feature_select_strategy=vision_feature_select_strategy,
            image_sizes=image_sizes,
        )
        image_features = torch.cat(image_features, dim=0).to(
            inputs_embeds.device, inputs_embeds.dtype
        )
        special_image_mask = self.get_placeholder_mask(
            input_ids, inputs_embeds=inputs_embeds, image_features=image_features
        )
        inputs_embeds = inputs_embeds.masked_scatter(special_image_mask, image_features)

    # Check if the image token exists in the current input_ids
    if (input_ids == self.config.image_token_id).any():
        image_limits = []
        for i in range(input_ids.shape[0]):
            indices = torch.where(input_ids[i] == self.config.image_token_id)[0]
            if indices.numel() > 0:
                image_limits.append((indices[0].item(), indices[-1].item()))
            else:
                image_limits.append(None)
            self.image_attention_mask = (input_ids != self.config.image_token_id).to(
                torch.int
            )
    else:
        image_limits = None
        if self.image_attention_mask is not None:
            self.image_attention_mask = torch.cat(
                (
                    self.image_attention_mask,
                    torch.ones(self.image_attention_mask.shape[:-1] + (1,)).to(
                        self.image_attention_mask.device
                    ),
                ),
                dim=-1,
            )
        else:
            self.image_attention_mask = None

    outputs = self.language_model(
        attention_mask=attention_mask,
        position_ids=position_ids,
        past_key_values=past_key_values,
        inputs_embeds=inputs_embeds,
        cache_position=cache_position,
        image_limits=image_limits,
        image_attention_mask=self.image_attention_mask,
        **kwargs,
    )

    return LlavaModelOutputWithPast(
        last_hidden_state=outputs.last_hidden_state,
        past_key_values=outputs.past_key_values,
        hidden_states=outputs.hidden_states,
        attentions=outputs.attentions,
        image_hidden_states=image_features if pixel_values is not None else None,
    )


def llama_forward(
    self,
    input_ids: torch.LongTensor | None = None,
    attention_mask: torch.Tensor | None = None,
    position_ids: torch.LongTensor | None = None,
    past_key_values: Cache | None = None,
    inputs_embeds: torch.FloatTensor | None = None,
    cache_position: torch.LongTensor | None = None,
    use_cache: bool | None = None,
    **kwargs: Unpack[TransformersKwargs],
) -> BaseModelOutputWithPast:
    image_limits = kwargs.get("image_limits", None)
    image_attention_mask = kwargs.get("image_attention_mask", None)
    if (input_ids is None) ^ (inputs_embeds is not None):
        raise ValueError("You must specify exactly one of input_ids or inputs_embeds")
    if inputs_embeds is None:
        inputs_embeds: torch.Tensor = self.embed_tokens(input_ids)

    if use_cache and past_key_values is None:
        past_key_values = DynamicCache(config=self.config)

    if cache_position is None:
        past_seen_tokens = (
            past_key_values.get_seq_length() if past_key_values is not None else 0
        )
        cache_position: torch.Tensor = (
            torch.arange(inputs_embeds.shape[1], device=inputs_embeds.device)
            + past_seen_tokens
        )

    if position_ids is None:
        position_ids = cache_position.unsqueeze(0)
    causal_mask = create_causal_mask(
        config=self.config,
        input_embeds=inputs_embeds,
        attention_mask=attention_mask,
        cache_position=cache_position,
        past_key_values=past_key_values,
        position_ids=position_ids,
    )
    print(causal_mask)
    image_mask = create_causal_mask(
        config=self.config,
        input_embeds=inputs_embeds,
        attention_mask=image_attention_mask,
        cache_position=cache_position,
        past_key_values=past_key_values,
        position_ids=position_ids,
    )

    def _mask_to_2d(mask: torch.Tensor, sample_idx: int) -> torch.Tensor:
        mask_2d = mask
        if mask_2d.dim() >= 4:
            mask_2d = mask_2d[sample_idx, 0]
        elif mask_2d.dim() == 3:
            mask_2d = mask_2d[sample_idx]
        return mask_2d.detach().to(torch.float32).cpu()
    
    if causal_mask is not None or image_mask is not None:
        # image_mask = torch.logical_and(image_mask, causal_mask)
        batch_size = image_mask.shape[0] if image_mask.dim() >= 3 else 1
        sample_count = min(2, batch_size)
        fig, axes = plt.subplots(sample_count, 2, figsize=(12, 5 * sample_count))
        if sample_count == 1:
            axes = [axes]

        for sample_idx in range(sample_count):
            # causal_plot = _mask_to_2d(causal_mask, sample_idx)
            image_plot = _mask_to_2d(image_mask, sample_idx)
            # im0 = axes[sample_idx][0].imshow(causal_plot, cmap="viridis", aspect="auto")
            # axes[sample_idx][0].set_title(f"Causal Mask (sample {sample_idx})")
            # axes[sample_idx][0].set_xlabel("Key position")
            # axes[sample_idx][0].set_ylabel("Query position")
            # fig.colorbar(im0, ax=axes[sample_idx][0]) # Add colorbar here

        
            im1 = axes[sample_idx][1].imshow(image_plot, cmap="viridis", aspect="auto")
            axes[sample_idx][1].set_title(f"Image Mask (sample {sample_idx})")
            axes[sample_idx][1].set_xlabel("Key position")
            axes[sample_idx][1].set_ylabel("Query position")
            fig.colorbar(im1, ax=axes[sample_idx][1]) # Add colorbar here

        plt.tight_layout()
        plt.show()

    hidden_states = inputs_embeds
    position_embeddings = self.rotary_emb(hidden_states, position_ids=position_ids)
    layers_to_skip = kwargs.get("layers_to_skip", [])
    if image_limits is not None:
        self.image_limits = image_limits
    for i, decoder_layer in enumerate(self.layers[: self.config.num_hidden_layers]):
        layer_mask = causal_mask
        if i in layers_to_skip and self.image_limits is not None:
            print("skipping layer", i)
            if i == layers_to_skip[0]:
                stored_embeds = hidden_states
            layer_mask = image_mask.clone()
            min_dtype = False
            for b, limit in enumerate(self.image_limits):
                if limit is not None:
                    start, end = limit
                    if layer_mask.shape[2] > 1:
                        layer_mask[b, :, end + 1 :, start : end + 1] = min_dtype
                    else:
                        layer_mask[b, :, :, start : end + 1] = min_dtype

        elif (
            len(layers_to_skip) > 0
            and i == (layers_to_skip[-1] + 1)
            and self.image_limits is not None
        ):
            for b, limit in enumerate(self.image_limits):
                if limit is not None:
                    start, end = limit
                    hidden_states[b, start : end + 1, :] = stored_embeds[
                        b, start : end + 1, :
                    ]
        hidden_states = decoder_layer(
            hidden_states,
            attention_mask=layer_mask,
            position_embeddings=position_embeddings,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            cache_position=cache_position,
            **kwargs,
        )
    hidden_states = self.norm(hidden_states)
    return BaseModelOutputWithPast(
        last_hidden_state=hidden_states,
        past_key_values=past_key_values,
    )


def patch1(layers_to_skip):
    LlamaModel.forward = partialmethod(llama_forward, layers_to_skip=layers_to_skip)
    LlavaModel.forward = llava_forward


In [ ]:
def load_image_from_url(url):
    try:
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        image = Image.open(BytesIO(response.content)).convert('RGB')
        return image
    except Exception as e:
        print(f"Error loading {url}: {e}")
        return None

# Tuple of (Image URL, Question)
data_samples = [
    (
        "http://images.cocodataset.org/val2017/000000039769.jpg",
        "Describe the scene in detail."
    ),
    (
        "http://images.cocodataset.org/val2017/000000000139.jpg",
        "What appliances can you see in this room?"
    )
]

In [ ]:
# Simplified inference
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image_from_url(url)
question = "Describe the scene in detail."

prompt = f"USER: <image>\n{question} ASSISTANT:"
inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

generate_ids = model.generate(**inputs, max_new_tokens=200)
print(processor.decode(generate_ids[0], skip_special_tokens=True))

In [ ]:
patch1(layers_to_skip=range(0))

In [ ]:
# Simplified inference
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = load_image_from_url(url)
question = "Describe the scene in detail."

prompt = f"USER: <image>\n{question} ASSISTANT:"
inputs = processor(text=prompt, images=image, return_tensors="pt").to(device)

generate_ids = model.generate(**inputs, max_new_tokens=2)
print(processor.decode(generate_ids[0], skip_special_tokens=True))

In [ ]:
urls = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "http://images.cocodataset.org/val2017/000000000139.jpg"
]
questions = [
    "Describe the scene in detail.",
    "What appliances can you see in this room?"
]

images = [load_image_from_url(url) for url in urls]
prompts = [f"USER: <image>\n{q} ASSISTANT:" for q in questions]

# Process batch with padding
inputs = processor(text=prompts, images=images, padding=True, return_tensors="pt").to(device)
generate_ids = model.generate(**inputs, max_new_tokens=2)
responses = processor.batch_decode(generate_ids, skip_special_tokens=True)

for response in responses:
    print(response.split("ASSISTANT:")[-1].strip())
    print("-" * 50)

In [ ]:
processor.decode(29871)

In [ ]:
urls = [
    "http://images.cocodataset.org/val2017/000000039769.jpg",
    "http://images.cocodataset.org/val2017/000000000139.jpg"
]
questions = [
    "Describe the scene in detail.",
    "What appliances can you see in this room?"
]

images = [load_image_from_url(url) for url in urls]
prompts = [f"USER: <image>\n{q} ASSISTANT:" for q in questions]

# Process batch with padding
inputs = processor(text=prompts, images=images, padding=True, return_tensors="pt").to(device)

generate_ids = model.generate(**inputs, max_new_tokens=1)
responses = processor.batch_decode(generate_ids, skip_special_tokens=True)

for response in responses:
    print(response.split("ASSISTANT:")[-1].strip())
    print("-" * 50)